# Week 5 — Theory
## Local interpretability and feature attribution

We have a trained model $f$ and an input $x$. The **local attribution problem** asks:

> *"Which features of $x$ contributed how much to the prediction $f(x)$?"*

This notebook covers the three method families you will use in the lab:

1. **Shapley values** (SHAP). Axiomatic, model-agnostic, expensive in the general case
   but exact and fast on trees.
2. **Local surrogate models** (LIME). Fit a simple, interpretable model in a
   neighbourhood of $x$ and report its coefficients.
3. **Gradient-based methods** (Integrated Gradients, DeepLIFT, Saliency). Cheap, only
   for differentiable models, sensitive to baseline choice.

We close with **sanity checks** — the failure modes attribution methods are known to
have, and how to test for them before trusting a published figure.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
plt.style.use("../../assets/mplstyle/course.mplstyle")
RNG = np.random.default_rng(0)


## 1. The Shapley value

Lloyd Shapley (1953) studied a cooperative-game problem: how to fairly divide a
collective payoff among players who contribute different amounts. The answer he proved
**unique under four axioms** turned out to be exactly what we want for feature attribution.

### The setup

Let $N = \{1, \ldots, n\}$ be the feature indices, and let
$v: 2^N \to \mathbb{R}$ be a *value function* — for a subset $S \subseteq N$, $v(S)$ is
"what the model would predict if it only had access to features in $S$". The Shapley
value of feature $i$ is:

$$
\phi_i \;=\; \sum_{S \subseteq N \setminus \{i\}}
\frac{|S|!\,(n - |S| - 1)!}{n!}\;\bigl[v(S \cup \{i\}) - v(S)\bigr]
$$

This is the **average marginal contribution** of feature $i$ to all possible
coalitions $S$ that don't already contain it.

### The four axioms

A function $\phi$ is the Shapley value iff it satisfies:

1. **Efficiency.** $\sum_i \phi_i = v(N) - v(\emptyset)$ — the attributions sum exactly
   to the prediction (minus the baseline). This is the property that makes SHAP plots
   look like budgets that balance.
2. **Symmetry.** If $v(S \cup \{i\}) = v(S \cup \{j\})$ for every $S$, then
   $\phi_i = \phi_j$. Two features that play identical roles get identical attributions.
3. **Dummy.** If $v(S \cup \{i\}) = v(S)$ for every $S$, then $\phi_i = 0$. A feature
   that the model ignores gets zero attribution.
4. **Additivity.** $\phi(v + w) = \phi(v) + \phi(w)$. Attributions for an ensemble are
   the sum of the per-member attributions.

These four properties together pin down a unique function. No other attribution method
satisfies all four exactly.

### What is $v(S)$ in practice?

The model is not designed to accept partial inputs. SHAP defines $v(S)$ as a conditional
expectation over the features not in $S$:

$$
v(S) = \mathbb{E}\!\left[f(X) \mid X_S = x_S\right]
$$

Computing this expectation is **the hard part**. Three practical strategies:

- **TreeExplainer** (Lundberg et al., 2018): exact in $O(\text{tree size} \cdot \text{features}^2)$
  for tree ensembles by exploiting the tree structure. Use it whenever your model is
  trees.
- **DeepExplainer**: approximates the expectation by averaging over a background dataset
  and using a DeepLIFT-style propagation. Use for deep networks.
- **KernelExplainer**: model-agnostic, samples random coalitions. Slow but works on
  anything.

### Caveat — independence and the conditional expectation

The standard SHAP implementation replaces "marginalise out" with "sample from the
marginal distribution of the absent features, treating them as independent of the
present ones". When features are correlated this is **not** the same as a true
conditional expectation, and can attribute non-zero importance to features the model
never actually used. The `interventional` vs. `tree_path_dependent` options in
TreeExplainer trade these properties off — read the SHAP docs carefully before
publishing a figure.


## 2. LIME — local surrogate models

LIME (Ribeiro et al., 2016) takes a different route. Around the input $x$, it
generates **perturbations** $z'$ (e.g. by zeroing out individual features or tokens),
computes $f(z')$ for each, and **fits a sparse linear model** $g$ on the perturbation
features, weighted by proximity to $x$:

$$
\hat g = \arg\min_{g \in G}\;
\sum_{z'} \pi_x(z')\,\bigl(f(z') - g(z')\bigr)^2 \;+\; \Omega(g)
$$

where $G$ is a family of interpretable models (usually linear regression on the
binarized perturbation space), $\pi_x$ is a proximity kernel, and $\Omega$ is a sparsity
penalty.

LIME's coefficients are the attributions. They are easier to interpret than raw
gradients because they live in a **human-meaningful feature space** (token presence,
super-pixel presence) rather than the raw input space.

### Trade-offs

- **Good.** Works on any model with `predict_proba`. Faster than KernelExplainer in
  practice. Conceptually transparent.
- **Bad.** The choice of perturbation kernel, the proximity kernel, and the number of
  samples are all hyperparameters. Re-run LIME twice on the same input and you may get
  different attributions — instability is the most-reported weakness.
- **Ugly.** LIME does not satisfy efficiency. The attributions do **not** sum to the
  prediction; they sum to whatever the local linear fit happens to predict at $x$.


## 3. Integrated Gradients and friends

For differentiable models there is a class of cheap, axiomatic methods. **Integrated
Gradients** (Sundararajan et al., 2017) defines the attribution of feature $i$ as the
path integral of the gradient from a *baseline* $x'$ to the input $x$:

$$
\mathrm{IG}_i(x) \;=\; (x_i - x'_i)\,
\int_0^1 \frac{\partial f\bigl(x' + \alpha(x - x')\bigr)}{\partial x_i}\, d\alpha
$$

In practice this integral is computed with a Riemann sum over, say, 50 steps. IG
satisfies two axioms that gradients alone do not:

- **Completeness:** $\sum_i \mathrm{IG}_i(x) = f(x) - f(x')$. The attributions sum to
  the difference between the prediction at $x$ and at the baseline.
- **Implementation invariance:** if two networks compute the same function, their IG
  attributions are identical.

Plain gradients ($\partial f / \partial x_i$) are saturated and very noisy on real
networks. **Use IG, not raw gradients.**

### The baseline matters more than people admit

IG is *relative to* a baseline $x'$. Different baselines give different attributions,
and there is no universal "neutral" baseline:

- **Black image** (zero pixels) is the canonical vision baseline, but a network that
  has never seen a fully-black image will produce garbage gradients along the
  interpolation path.
- **Blurred image** is a more honest neutral but loses spatial information.
- **Token <pad>** is the canonical NLP baseline, but the embedding for <pad> may not
  be small.
- **Average of training data** is a defensible "what is the typical input?" baseline.

A published IG figure that does not state its baseline is not reproducible. Always
report it.

### Related Captum methods

- **DeepLIFT.** A more efficient discrete-step approximation to IG; same axioms,
  cheaper to compute.
- **Saliency.** Just the absolute value of $\partial f / \partial x$. Quick, noisy,
  fails axioms — use only as a sanity check.
- **GuidedBackprop / SmoothGrad / Layer-wise variants.** Useful when you specifically
  want layer-wise attribution; we'll meet some of these in week 6.


## 4. Sanity checks for attribution methods

Adebayo et al. (2018) showed that **several popular attribution methods produce visually
similar maps even for randomly-initialized networks**. If a method's output does not
change when the model's parameters are randomized, that method is not actually
explaining the model — it's mostly responding to the input.

The two sanity checks they propose:

### Model parameter randomization test

Replace the trained weights with random ones (layer by layer, top-down) and re-compute
the attributions. If they don't change, the method is dominated by the input statistics
and is **not faithful** to the model.

### Data randomization test

Train the model on labels that have been randomly permuted, so the model can only
memorize. Re-compute the attributions. If they look "sensible", the method is again
not really telling you anything about the function being computed.

**Pass:** Integrated Gradients, SHAP (with the right value function), DeepLIFT.
**Fail (in their experiments):** vanilla saliency, GuidedBackprop, GuidedGradCAM.

We will demonstrate one of these checks in the week-6 lab on a CNN.

### Faithfulness metrics

A quantitative alternative: pick the top-K features by attribution, **remove them**
(e.g. replace with baseline), and measure how much $f(x)$ drops. A faithful attribution
method shows a steep drop; an unfaithful one shows a shallow drop. This is the basis of
**perturbation-based faithfulness scores** that we use in week 6.


## 5. When to use which method

A quick decision tree.

```
Is the model a tree ensemble (XGBoost, LightGBM, RandomForest)?
  → SHAP TreeExplainer. Exact, fast, no baseline question.

Is the model differentiable and small?
  → Captum Integrated Gradients with a defensible baseline.

Is the model a black box (API, ensemble of unknown type)?
  → SHAP KernelExplainer for global views,
    LIME for fast per-instance human-readable views.

Is the input text (tokens)?
  → LIME or SHAP on token-presence features.

Is the input an image and the model a CNN/ViT?
  → Layer-wise methods covered in week 6 (Grad-CAM, attention rollout) rather than
    raw input attribution.

Is the input multimodal?
  → IG / SHAP per modality, then sum modality-level attributions for a budget.
```

A single method is rarely enough for a published claim. **Triangulate**: report at
least one global method (SHAP summary) and one local method on a few representative
instances, and disclose the sanity-check results.


## Summary

- The **Shapley value** is the unique attribution that satisfies efficiency, symmetry,
  dummy, and additivity.
- **SHAP** estimates it; choose the explainer to match your model.
- **LIME** is a fast, model-agnostic alternative built on local linear surrogates;
  it is **unstable** and does not satisfy efficiency.
- **Integrated Gradients** is the right default for differentiable models. The
  **baseline** choice changes the attribution; always report it.
- **Run sanity checks.** A method that produces the same map for a trained and a
  randomly-initialized model is not explaining your model.

In the lab we walk through three concrete attributions: SHAP on a tabular regressor,
LIME on a text classifier, and IG (Captum) on a PyTorch CNN.
